# Why didn't the 2017 models work on 2018?

Models trained on CIC-IDS2017 caught almost none of the attacks in CSE-CIC-IDS2018
(`reports/cross_dataset_2018.md`), including attack types they were trained on. This notebook
checks the possible causes one at a time:

1. **Format.** Do the columns mean the same thing and use the same units?
2. **Cleaning.** Did the cleaning pipeline treat the two years differently?
3. **Drift.** How different is ordinary benign traffic between the two years?
4. **Traffic or tool?** Is the difference in the network traffic, or in how the flow tool measured it?
5. **Same attack, different year.** Do the shared attack types look alike?
6. **Class mix.** Which attack types exist in only one year?

In [1]:
import sys
sys.path.insert(0, "..")  # make src/ importable from notebooks/

import json
import warnings
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier

from src.models.cross_dataset import FAMILY, load_year

pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", message="X does not have valid feature names")
SEED = 42

df17, y17, _ = load_year("2017")
df18, y18, _ = load_year("2018")
features = df17.select_dtypes(include=[np.number]).columns.tolist()
assert features == df18.select_dtypes(include=[np.number]).columns.tolist()
print(f"2017: {len(df17):,} rows | 2018: {len(df18):,} rows | {len(features)} shared features")

2017: 2,519,262 rows | 2018: 2,514,216 rows | 71 shared features


## 1. Format

The 2018 CSVs use abbreviated column names. `src/data/cicids2018.py` maps them onto the 2017
names; the mapping was checked by position against both raw headers (78 of 78 match), and
`tests/test_cicids2018.py` covers the loader. Two more checks: are time features in the same
units, and is any feature constant in one year only?

In [2]:
time_cols = ["Flow Duration", "Flow IAT Max", "Fwd IAT Max", "Active Max", "Idle Max"]
pd.DataFrame({"2017 max": df17[time_cols].max(), "2018 max": df18[time_cols].max()})

,2017 max,2018 max
Flow Duration,119999998,119999998.0
Flow IAT Max,120000000,120000000.0
Fwd IAT Max,120000000,120000000.0
Active Max,110000000,114000000.0
Idle Max,120000000,120000000.0


In [3]:
constant = [f for f in features if min(df17[f].nunique(), df18[f].nunique()) <= 1]
print("Features constant in one year:", constant or "none")

Features constant in one year: none


Both years cap flows at 120 seconds (120,000,000 microseconds), so time features share units.
No feature is constant in either year. The columns line up.

## 2. Cleaning

In [4]:
m17 = json.load(open("../reports/dataset_v2_manifest.json"))
m18 = json.load(open("../reports/dataset_2018_manifest.json"))
steps = ["rows_raw", "rows_dropped_inf_rates", "rows_dropped_negative",
         "rows_dropped_duplicates", "rows_dropped_contradictory_labels", "rows_clean"]
cleaning = pd.DataFrame({"2017": [m17[s] for s in steps], "2018": [m18[s] for s in steps]}, index=steps)
cleaning.loc["duplicates as share of raw"] = (cleaning.loc["rows_dropped_duplicates"] / cleaning.loc["rows_raw"]).round(3)
cleaning.loc["sentinel flows as share of clean"] = [
    round(m17["sentinel_flows_kept"] / m17["rows_clean"], 3), round(m18["sentinel_flows_kept"] / m18["rows_clean"], 3)]
cleaning

,2017,2018
rows_raw,2830743.000,4096716.000
rows_dropped_inf_rates,2867.000,10681.000
rows_dropped_negative,150.000,3.000
rows_dropped_duplicates,307070.000,1557465.000
rows_dropped_contradictory_labels,1394.000,14351.000
rows_clean,2519262.000,2514216.000
duplicates as share of raw,0.108,0.380
sentinel flows as share of clean,0.571,0.855


The same code cleaned both years. Two differences come from the data, not the pipeline:
2018 has far more exact duplicates, and 2018 `rows_raw` counts only the 10% benign sample (every
attack flow was kept), so its shares are not directly comparable to 2017's. Cleaning also nearly
removes two 2018 classes (next cell).

In [5]:
pd.Series(m18["label_counts"]).loc[["FTP-BruteForce", "DoS attacks-SlowHTTPTest"]].rename("2018 rows after cleaning")

FTP-BruteForce              39
DoS attacks-SlowHTTPTest    43
Name: 2018 rows after cleaning, dtype: int64

## 3. How far did ordinary traffic drift?

We compare **benign** flows only, so differences in the attack mix can't explain the result.
For each feature, the Kolmogorov-Smirnov statistic is the largest gap between the two years'
cumulative distributions: 0 means identical, 1 means no overlap at all.

In [6]:
benign17 = df17[y17 == 0].sample(200_000, random_state=SEED)
benign18 = df18[y18 == 0].sample(200_000, random_state=SEED)
ks = pd.Series({f: ks_2samp(benign17[f], benign18[f]).statistic for f in features}).sort_values(ascending=False)
print(f"Median KS across features: {ks.median():.3f}; features with KS > 0.1: {(ks > 0.1).sum()} of {len(ks)}")
ks.head(15).round(3).to_frame("KS statistic")

Median KS across features: 0.174; features with KS > 0.1: 53 of 71


,KS statistic
min_seg_size_forward,0.433
Fwd Packet Length Min,0.326
Min Packet Length,0.318
Bwd Header Length,0.310
Fwd IAT Min,0.303
Fwd IAT Total,0.295
Fwd IAT Max,0.294
Fwd IAT Mean,0.292
Bwd Packet Length Min,0.287
Flow IAT Min,0.284


**Adversarial validation.** A sharper test: train a classifier to guess the *year* of a benign
flow. If it can't (AUC near 0.5), the two years look alike. If it can, the model has something
year-specific to latch onto. We then remove the most-shifted features and try again.

In [7]:
X = pd.concat([benign17[features], benign18[features]]).values
year = np.r_[np.zeros(len(benign17)), np.ones(len(benign18))]
X_tr, X_te, yr_tr, yr_te = train_test_split(X, year, test_size=0.3, stratify=year, random_state=SEED)

def year_auc(cols):
    idx = [features.index(c) for c in cols]
    clf = LGBMClassifier(n_estimators=200, random_state=SEED, verbose=-1).fit(X_tr[:, idx], yr_tr)
    return clf, roc_auc_score(yr_te, clf.predict_proba(X_te[:, idx])[:, 1])

clf, auc_all = year_auc(features)
gain = pd.Series(clf.booster_.feature_importance("gain"), index=features)
print(f"Year classifier AUC with all {len(features)} features: {auc_all:.4f}")
(gain / gain.sum()).sort_values(ascending=False).head(8).round(3).to_frame("share of gain")

Year classifier AUC with all 71 features: 0.9998


,share of gain
min_seg_size_forward,0.450
Fwd Packet Length Min,0.113
Max Packet Length,0.079
Min Packet Length,0.049
Fwd Packet Length Std,0.040
Destination Port,0.035
Fwd IAT Min,0.030
Flow IAT Min,0.023


In [8]:
pd.DataFrame([
    {"features dropped": n, "features left": len(features) - n,
     "year classifier AUC": round(year_auc([f for f in features if f not in ks.index[:n]])[1], 4)}
    for n in [0, 10, 30, 50, 65]
]).set_index("features dropped")

,features left,year classifier AUC
features dropped,,
0,71,0.9998
10,61,0.9997
30,41,0.9945
50,21,0.9470
65,6,0.5165


## 4. Is it the traffic, or the tool?

`min_seg_size_forward` shifts the most and dominates the year classifier. A clean way to test
whether that is a traffic difference or a measurement difference: compare the **same kind of flow**
in both years. Single-packet DNS queries (port 53, UDP, no TCP window) should look identical on
any network. A UDP header is 8 bytes.

In [9]:
rows = {}
for name, d in [("2017", df17), ("2018", df18)]:
    dns = d[(d["Label"] == "BENIGN") & (d["Destination Port"] == 53) & (d["Total Fwd Packets"] == 1)]
    rows[name] = {
        "single-packet DNS flows": len(dns),
        "has TCP window": dns["Has_Init_Win_fwd"].mean().round(3),
        "min_seg_size_forward (top values)": dns["min_seg_size_forward"].value_counts(normalize=True).head(3).round(3).to_dict(),
        "Fwd Header Length (top values)": dns["Fwd Header Length"].value_counts(normalize=True).head(3).round(3).to_dict(),
    }
pd.DataFrame(rows)

,2017,2018
single-packet DNS flows,252169,220752
has TCP window,0.0,0.0
min_seg_size_forward (top values),"{32: 0.48, 20: 0.452, 40: 0.056}","{8: 1.0, 24: 0.0, 20: 0.0}"
Fwd Header Length (top values),"{32: 0.48, 20: 0.452, 40: 0.056}","{8: 1.0, 24: 0.0, 20: 0.0}"


## 5. Same attack, different year

Medians of a few features for attack types that appear in both years, next to benign traffic.

In [10]:
pairs = [("SSH-Patator", "SSH-Bruteforce"), ("DoS Hulk", "DoS attacks-Hulk"),
         ("DoS slowloris", "DoS attacks-Slowloris"), ("Bot", "Bot"), ("BENIGN", "BENIGN")]
cols = ["Destination Port", "Flow Duration", "Total Fwd Packets", "Total Backward Packets",
        "Average Packet Size", "Init_Win_bytes_forward"]
medians = {}
for l17, l18 in pairs:
    medians[f"2017 {l17}"] = df17.loc[df17["Label"] == l17, cols].median()
    medians[f"2018 {l18}"] = df18.loc[df18["Label"] == l18, cols].median()
pd.DataFrame(medians).T.round(1)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Average Packet Size,Init_Win_bytes_forward
2017 SSH-Patator,22.0,12029788.0,21.0,32.0,89.7,29200.0
2018 SSH-Bruteforce,22.0,374609.0,22.0,22.0,104.8,26883.0
2017 DoS Hulk,80.0,86419094.0,6.0,6.0,918.2,251.0
2018 DoS attacks-Hulk,80.0,50949.0,3.0,0.0,0.0,225.0
2017 DoS slowloris,80.0,99999407.0,3.0,2.0,6.0,29200.0
2018 DoS attacks-Slowloris,80.0,106338597.0,4.0,2.0,34.0,26883.0
2017 Bot,8080.0,71035.5,3.0,3.0,9.0,8192.0
2018 Bot,8080.0,10909.0,3.0,4.0,65.0,8192.0
2017 BENIGN,80.0,40277.0,2.0,2.0,78.2,122.0
2018 BENIGN,443.0,149420.0,3.0,2.0,87.0,980.0


## 6. Class mix

In [11]:
fam = pd.DataFrame({
    "2017 flows": df17["Label"].map(FAMILY).value_counts(),
    "2018 flows (benign is a 10% sample)": df18["Label"].map(FAMILY).value_counts(),
}).fillna(0).astype(int)
fam

,2017 flows,2018 flows (benign is a 10% sample)
Label,,
Benign,2094218,1170047
Bot,1948,144535
Brute force (FTP/SSH),9150,94080
DDoS,128011,775955
DoS,193615,196556
Heartbleed,11,0
Infiltration,36,132188
PortScan,90130,0
Web attack,2143,855


## Summary

The failure is not a column-name or cleaning problem. It comes from three things, in order of how
clearly the data shows them.

**1. The flow tool measured some features differently.** For single-packet DNS queries, the same
kind of flow in both years, 2018 records a forward header length of 8 bytes in 100% of flows. That
is the real size of a UDP header. 2017 records 20, 32 or 40 bytes. The feature has the same name
but a different meaning. `min_seg_size_forward`, the most-shifted feature (KS 0.433) and 45% of
the year classifier's gain, is one of these header-based features.

**2. The networks and attack tools differ.** Benign traffic shifted on 53 of 71 features, and its
typical destination port moved from 80 to 443. A classifier can tell which year a benign flow came
from with AUC 0.9998. It still gets 0.947 using only the 21 least-shifted features, and only drops
to 0.517 when 6 features are left. The same attack names also hide different behaviour. 2018's SSH
brute-force flows are about 32 times shorter. 2018's DoS Hulk flows get no reply packets and have
an average packet size of 0, against 918 bytes in 2017. Not everything moved, though: slowloris
lasts about 100 seconds in both years, and Bot uses port 8080 with an 8,192-byte TCP window in
both.

**3. The class mix differs.** 2018 has no PortScan or Heartbleed. 2017 has only 36 Infiltration
and 1,948 Bot flows, against 132,188 and 144,535 in 2018. Cleaning nearly removes 2018's
FTP-BruteForce (39 rows) and SlowHTTPTest (43 rows), because 38% of the loaded 2018 rows (all attacks, 10% of benign) were exact
duplicates, against 10.8% in 2017.

**What the checks ruled out.** Units match: both years cap flows at 120 seconds. All 78 columns
map one-to-one, no feature is constant in either year, and the same cleaning code ran on both.

**What to try next** (`train_2017_test_2018.ipynb`, `train_2018_test_2017.ipynb`): drop the
features the tool or network sets, normalize each year against its own traffic, and see whether
either brings back the attacks that look alike across years.